# Qlib on Google Colab: Full Research Guide

This notebook is a gentle, hands-on tour of **[Qlib](https://github.com/microsoft/qlib)** for quant research. It is written so even a 5-year-old can follow the big ideas.


## How to use this notebook
- Run each code cell in order on **Google Colab** (GPU is optional; CPU works).
- Most cells are light-weight; data downloads can take a few minutes.
- Words in *italic* are simplified explanations.


## PART 1 — Overview of Qlib (Explain Like I'm 5)
- **What is Qlib?** It is a big box of tools that helps us study how stocks move. Think of it as a LEGO kit for finance.
- **Why do quant researchers use it?** It saves time. Instead of building everything from scratch, Qlib gives ready-made blocks for data, models, and backtests.
- **Key concepts:**
  - **Market data:** prices and volumes for stocks—like daily pictures of how a stock is doing.
  - **Factors / features:** simple numbers we compute from data (e.g., yesterday's return). They are like clues.
  - **Labels:** the answer we want to predict (e.g., future return).
  - **Models:** machines that learn patterns from clues to guess labels.
  - **Alpha / signal / score:** the model's guess about how good a stock might be soon. Higher = better.
  - **Backtesting:** pretend trading in the past to see if our ideas make money.


## PART 2 — Installation & Setup
Below are Colab-friendly cells to install and initialize Qlib.


In [ ]:
# Install Qlib from GitHub (latest)
!pip install -qU pip
!pip install -q git+https://github.com/microsoft/qlib.git
# Optional: common extras for models/backtests
!pip install -q lightgbm matplotlib seaborn pandas numpy pytorch-lightning


In [ ]:
# Import and show version
import qlib
print("Qlib version:", qlib.__version__)


In [ ]:
# Initialize Qlib with **local data** (downloaded later). Region "cn" for China A-shares.
from qlib import init
from qlib.config import REG_CN, REG_US

init(provider_uri="~/.qlib/qlib_data/cn_data", region=REG_CN)


In [ ]:
# Initialize Qlib with **remote provider** (built-in cloud mirror). Good when you don't want to store data locally.
# If the remote mirror is temporarily unreachable, we fall back to local data (see download cell above).
import qlib
from pathlib import Path
from qlib.config import REG_CN

REMOTE_URI = "http://qlib.oss-cn-shenzhen.aliyuncs.com/qlib/qlib_data/cn_data"
LOCAL_URI = Path("~/.qlib/qlib_data/cn_data").expanduser().as_posix()

try:
    qlib.init(provider_uri=REMOTE_URI, region=REG_CN)
    print("Remote provider initialized:", REMOTE_URI)
except Exception as e:
    print("Remote provider failed (", e, ") — falling back to local cache", sep="")
    qlib.init(provider_uri=LOCAL_URI, region=REG_CN)
    print("Local provider initialized:", LOCAL_URI)


## PART 3 — Data Processing (Qlib Building Blocks)

### Dataset initialization
Qlib uses `Dataset` objects to bundle **features** (clues) and **labels** (answers) with **instruments** (stock list) and **calendar** (dates).

### DataHandler (ELI5)
A `DataHandler` is like a little chef. It reads raw data (rice), cooks it (cleans and normalizes), and serves tidy tables (ready-to-eat). You tell it which dishes (features/labels) you want.

### How Qlib structures data
- **Instruments:** which stocks, e.g., CSI300 tickers (`csi300`) or S&P500 (`sp500`).
- **Calendar:** list of trading days.
- **Features:** expressions like `$close/Ref($close,1)-1` (yesterday's return).
- **Labels:** targets like `Ref($close, -2)/Ref($close, -1)-1` (next-day return).


In [ ]:
# Download daily China A-share data to ~/.qlib/qlib_data/cn_data (~1-2GB)
!python -m qlib.run.get_data qlib_data --target_dir ~/.qlib/qlib_data/cn_data --region cn


In [ ]:
# Download daily US data to ~/.qlib/qlib_data/us_data (~1GB)
!python -m qlib.run.get_data qlib_data --target_dir ~/.qlib/qlib_data/us_data --region us


In [ ]:
# Load raw data and peek
import qlib
from qlib.data import D
qlib.init(provider_uri="~/.qlib/qlib_data/cn_data", region=qlib.config.REG_CN)

# Choose instruments and dates
market = "csi300"
start, end = "2020-01-01", "2020-03-01"
fields = ["$close", "$volume"]

df_raw = D.features(instruments=market, fields=fields, start_time=start, end_time=end)
df_raw.head()


In [ ]:
# Normalize data (z-score by instrument)
from qlib.data.filter import NameDFilter
from qlib.data import D
import pandas as pd

qlib.init(provider_uri="~/.qlib/qlib_data/cn_data", region=qlib.config.REG_CN)

fields = ["$close", "$volume"]
raw = D.features("csi300", fields, start_time="2020-01-01", end_time="2020-12-31")
normalized = raw.groupby(level=0).apply(lambda x: (x - x.mean()) / x.std()).dropna()
normalized.head()


In [ ]:
# Feature engineering examples
feature_config = {
    "feature": [
        "$close/Ref($close,1)-1",          # daily return
        "Mean($close,5)/Ref($close,1)-1",  # 5-day momentum
        "Std($close,20)",                  # 20-day volatility proxy
        "$volume/Mean($volume,20)",        # volume surge ratio
    ],
    "label": [
        "Ref($close,-2)/Ref($close,-1)-1"  # next-day return
    ],
}
feature_config


In [ ]:
# Build a Dataset with DataHandlerLP (Label & Price handler)
from qlib.data.dataset import DatasetH
from qlib.data.dataset.handler import DataHandlerLP
from qlib.contrib.data.handler import Alpha158

# Alpha158 gives many ready-made factors; you can also pass custom expressions
handler = DataHandlerLP(
    instruments="csi300",
    start_time="2018-01-01",
    end_time="2020-12-31",
    fit_start_time="2018-01-01",
    fit_end_time="2019-12-31",
    infer_processors=[{"class": "RobustZScoreNorm", "kwargs": {"fields_group": "feature"}}],
    learn_processors=[{"class": "DropnaLabel"}],
    windows=feature_config,
)

dataset = DatasetH(handler)
print(dataset)


In [ ]:
# Plot example data (closing prices for one stock)
import matplotlib.pyplot as plt

one_stock = df_raw.xs("SH600000")
one_stock["$close"].plot(title="SH600000 Close Price")
plt.xlabel("Date")
plt.ylabel("Price")
plt.show()


## PART 4 — Modeling

### Model zoo (what is available?)
- **Linear models:** simple straight-line brains (`LinearModel`).
- **Tree models:** smarter branching brains (`LGBModel`, `CatBoostModel`).
- **DNN models:** deep learning brains (`GRUModel`, `GATs`, `ALSTM`, `Transformer`, etc.).

### Hyperparameters (ELI5)
- They are the knobs on a toy car (speed, steering). We turn them to make the model fit better.
- Examples: number of trees, learning rate, hidden size, epochs.

### Train/valid/test split
- **Train:** model learns.
- **Valid:** we check if it is learning well and tune knobs.
- **Test:** final exam on unseen data.


In [ ]:
# Define dataset splits
from qlib.data.dataset.handler import DataHandlerLP
from qlib.data.dataset import DatasetH

handler = DataHandlerLP(
    instruments="csi300",
    start_time="2015-01-01",
    end_time="2020-12-31",
    fit_start_time="2015-01-01",
    fit_end_time="2018-12-31",
    infer_processors=[{"class": "RobustZScoreNorm", "kwargs": {"fields_group": "feature"}}],
    learn_processors=[{"class": "DropnaLabel"}],
    windows={"feature": ["$close/Ref($close,1)-1"], "label": ["Ref($close,-2)/Ref($close,-1)-1"]},
)

dataset = DatasetH(handler)
conf = {
    "train": {"start_time": "2015-01-01", "end_time": "2018-12-31"},
    "valid": {"start_time": "2019-01-01", "end_time": "2019-12-31"},
    "test": {"start_time": "2020-01-01", "end_time": "2020-12-31"},
}
conf


In [ ]:
# Train a baseline Linear model
from qlib.contrib.model.linear import LinearModel
from qlib.contrib.workflow import R

model = LinearModel(loss="mse")
linear_recorder = R.train(model=model, dataset=dataset, recorder_name="linear_baseline")


In [ ]:
# Train a LightGBM model
from qlib.contrib.model.gbdt import LGBModel

lgb_params = {
    "num_leaves": 64,
    "learning_rate": 0.05,
    "feature_fraction": 0.8,
    "n_estimators": 200,
}

lgb_model = LGBModel(loss="mse", **lgb_params)
lgb_recorder = R.train(model=lgb_model, dataset=dataset, recorder_name="lgb_baseline")


In [ ]:
# Train a GRU deep model (sequence model)
from qlib.contrib.model.pytorch_gru import GRUModelPytorch

gru_model = GRUModelPytorch(
    d_feat=1,             # feature dimension
    hidden_size=16,       # knob: how many hidden neurons
    num_layers=2,
    dropout=0.1,
    epochs=5,             # small for demo; increase for real runs
)
gru_recorder = R.train(model=gru_model, dataset=dataset, recorder_name="gru_demo")


In [ ]:
# Save and load a trained model
import pickle, os

model_path = "./saved_qlib_model.pkl"
with open(model_path, "wb") as f:
    pickle.dump(lgb_model, f)
print("Saved to", model_path)

with open(model_path, "rb") as f:
    loaded_model = pickle.load(f)
print("Loaded model type:", type(loaded_model))


## PART 5 — Forecasting / Inference
- **Alpha / signal / score:** the model's number saying "I think this stock will go up (high) or down (low)."
- We feed test data to the model to get these scores.


In [ ]:
# Prepare test data and predict (recorder-free to avoid path errors)
# We call the model directly so no recorder URI is needed.

test_data = dataset.prepare("test")
raw_pred = loaded_model.predict(dataset=dataset, segment="test")

# Build a DataFrame with scores and labels for later analysis
pred_df = raw_pred.rename("score").to_frame()
pred_df["label"] = test_data.get_label().loc[pred_df.index]

pred_df.head()


In [ ]:
# Save predictions to CSV
pred_path = "./predictions.csv"
pred_df.to_csv(pred_path)
print("Predictions saved to", pred_path)


In [ ]:
# Visualize prediction distribution
import matplotlib.pyplot as plt

pred_df["score"].hist(bins=50)
plt.title("Prediction Score Distribution")
plt.xlabel("Score")
plt.ylabel("Count")
plt.show()


## PART 6 — Portfolio & Backtest
- **Risk model:** estimates how bumpy the road is (volatility).
- **Execution model:** simulates how orders fill; simple one just buys/sells at close price.
- **Position management:** how much to buy/sell.
- **Benchmark:** yardstick like CSI300 or SP500.
- **TrainerFlow:** pipeline that glues training + validation + backtest.
- **Words (ELI5):**
  - **Sharpe ratio:** like score per unit of noise—higher is better.
  - **Drawdown:** biggest drop from a high point—how deep the valley gets.
  - **Turnover:** how often we trade—like how many toy swaps we do.


In [ ]:
# Simple backtest using TopK strategy and Naive executor
from qlib.contrib.strategy.signal_strategy import TopkDropoutStrategy
from qlib.contrib.executor.executor import NaiveExecutor
from qlib.contrib.evaluator import backtest

# Use predictions from pred_df
strategy = TopkDropoutStrategy(signal=pred_df, topk=10, n_drop=2)
executor = NaiveExecutor()

report, positions = backtest(strategy=strategy, executor=executor, start_time="2020-01-01", end_time="2020-12-31", benchmark="SH000300")
print(report.tail())


In [ ]:
# Plot portfolio equity curve
import matplotlib.pyplot as plt

equity = report["return"].add(1).cumprod()
equity.plot(title="Equity Curve")
plt.xlabel("Date")
plt.ylabel("Portfolio Value (1=start)")
plt.show()


## PART 7 — Performance Analysis
Metrics and how to read them (ELI5):
- **Prediction IC (Information Coefficient):** correlation between predictions and reality. Like how often guesses align with truth.
- **ICIR:** average IC divided by its wiggles. Steadier is better.
- **Cumulative return:** how money grows over time.
- **Annualized return:** yearly growth speed.
- **Risk metrics:** volatility (how bumpy), max drawdown (deepest dip), turnover (how often we trade).
- **Confusion-matrix-like alpha check:** how many top-ranked stocks actually did well.
- **Feature importance (trees):** which clues mattered most.


In [ ]:
# Compute IC/ICIR
import numpy as np

ic = pred_df.groupby(level=0).apply(lambda x: x["score"].corr(x["label"], method="spearman"))
icir = ic.mean() / ic.std()
print("IC mean:", ic.mean(), "ICIR:", icir)


In [ ]:
# Basic return stats
cumulative = equity.iloc[-1] - 1
annualized = (equity.iloc[-1]) ** (252/len(equity)) - 1  # assuming 252 trading days
max_dd = (equity.cummax() - equity).max()
print("Cumulative return:", cumulative)
print("Annualized return:", annualized)
print("Max drawdown:", max_dd)


In [ ]:
# Confusion-matrix-like analysis: bucket by score
import pandas as pd

pred_df = pred_df.reset_index()
pred_df["bucket"] = pd.qcut(pred_df["score"], q=5, labels=False)
bucket_perf = pred_df.groupby("bucket")["label"].mean().sort_index(ascending=False)
print(bucket_perf)


In [ ]:
# Feature importance for LightGBM
import pandas as pd
import matplotlib.pyplot as plt

importance = pd.Series(lgb_model.model.feature_importances_)
importance.plot.bar(title="Feature Importance")
plt.show()


## PART 8 — Advanced Qlib Features
- **Online serving:** turn your model into an API; see `qlib/server` for templates.
- **Optimization:** hyperparameter search via `qlib.contrib.workflow.record_temp.Experiment`. You can plug in Optuna/Hyperopt.
- **Workflow pipelines:** `Trainer` and `TrainerFlow` connect data -> model -> backtest -> report automatically.
- **Custom DataHandlers:** subclass `DataHandlerLP` and override `setup_data` to load your own CSV/DB data.
- **Custom models:** subclass `Model` and implement `fit`, `predict`, and `save/load`. Minimal PyTorch template below.


In [ ]:
# Minimal custom model template
from qlib.model.base import Model
import torch

class MyTinyModel(Model):
    def __init__(self):
        super().__init__()
        self.linear = torch.nn.Linear(1, 1)

    def fit(self, dataset, **kwargs):
        data = dataset.prepare("train")
        X, y = data.get_all_features().values, data.get_label().values
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        optim = torch.optim.Adam(self.linear.parameters(), lr=1e-2)
        for _ in range(50):
            optim.zero_grad()
            loss = torch.nn.functional.mse_loss(self.linear(X).squeeze(), y)
            loss.backward()
            optim.step()

    def predict(self, dataset):
        data = dataset.prepare("test")
        X = torch.tensor(data.get_all_features().values, dtype=torch.float32)
        with torch.no_grad():
            pred = self.linear(X).squeeze().numpy()
        return data.get_label().index, pred

    def save(self, path):
        torch.save(self.linear.state_dict(), path)

    def load(self, path):
        self.linear.load_state_dict(torch.load(path))


## PART 9 — Summary & Next Steps
- You learned how to install Qlib, prepare data, train models, make forecasts, and backtest.
- **What to explore next:**
  - Try richer factors (Alpha360, Alpha158) and custom expressions.
  - Tune hyperparameters with more epochs/trees.
  - Experiment with US data (`REG_US`) and different universes (`sp500`, `nasdaq100`).
- **Docs:** https://qlib.readthedocs.io and https://github.com/microsoft/qlib
- **Contribute:** open issues/PRs, improve docs, or add models in `qlib/contrib/model/`.
- **Datasets to try:**
  - Qlib US daily (`~/.qlib/qlib_data/us_data`)
  - Your own CSVs via custom DataHandler
  - Crypto or ETFs (after formatting to Qlib schema)
